# Train the gesture model

Run these cells in order after `prepare_data.py`. The test set is used only after training is complete.

In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

week_dir = Path.cwd()
if not (week_dir / "prepared_data.npz").exists():
    candidate = week_dir / "week2-model-training"
    if (candidate / "prepared_data.npz").exists():
        week_dir = candidate
models_dir = week_dir / "models"
models_dir.mkdir(exist_ok=True)
data = np.load(week_dir / "prepared_data.npz")
labels = [str(value) for value in data["labels"]]
x_train, y_train = data["x_train"], data["y_train"]
x_validation, y_validation = data["x_validation"], data["y_validation"]
x_test, y_test = data["x_test"], data["y_test"]
print(x_train.shape, x_validation.shape, x_test.shape, labels)

The input keeps time and feature dimensions visible. `Flatten` lays those values in one row. The 48-unit ReLU layer learns combinations of motion values; softmax produces three probabilities.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 6)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(48, activation="relu"),
    tf.keras.layers.Dense(len(labels), activation="softmax"),
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Early stopping watches validation loss. It stops after eight epochs without improvement and restores the best weights rather than the last weights.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=80,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history["loss"], label="training")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch")
axes[0].legend()
axes[1].plot(history.history["accuracy"], label="training")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

Now, and only now, the untouched test set measures the final model. Rows in the confusion matrix are actual labels and columns are predictions.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
probabilities = model.predict(x_test, verbose=0)
predictions = probabilities.argmax(axis=1)
matrix = np.zeros((len(labels), len(labels)), dtype=int)
for actual, predicted in zip(y_test, predictions):
    matrix[int(actual), int(predicted)] += 1
per_class_recall = {
    label: float(matrix[index, index] / matrix[index].sum())
    if matrix[index].sum() else 0.0
    for index, label in enumerate(labels)
}
print(f"Test accuracy: {test_accuracy:.1%}")
print("Per-class recall:", per_class_recall)
print(matrix)

In [ ]:
figure, axis = plt.subplots(figsize=(5, 4))
image = axis.imshow(matrix, cmap="Blues")
axis.set(xticks=range(len(labels)), yticks=range(len(labels)),
         xticklabels=labels, yticklabels=labels,
         xlabel="Predicted", ylabel="Actual", title="Test confusion matrix")
for row in range(len(labels)):
    for column in range(len(labels)):
        axis.text(column, row, matrix[row, column], ha="center", va="center")
figure.colorbar(image, ax=axis)
plt.tight_layout()
figure.savefig(models_dir / "confusion_matrix.png", dpi=160)
plt.show()

model.save(models_dir / "gesture_model.keras")
metrics = {
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "confusion_matrix": matrix.tolist(),
    "per_class_recall": per_class_recall,
    "test_recordings": [str(value) for value in data["recording_test"]],
}
(models_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print(f"Saved final artifacts in {models_dir}")